In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import xgboost as xgb

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)  # one line per trial is too much output

RANDOM_STATE = 0
FAST = True  # TESTING: cheap grids/trial counts for quick iteration; set False for the full run

DATA = Path("data/processed")  # run from the repository root
tract = pd.read_csv(DATA / "sovi_tract_2015_merged.csv", dtype={"GEOID": str})

TARGET = "life_expectancy"
FEATURES = [c for c in tract.columns
            if c not in ("GEOID", "NAME", "population", TARGET) and not c.startswith("geo_")]
assert len(FEATURES) == 28

n_before = len(tract)
tract = tract.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
print(f"Dropped {n_before - len(tract):,} of {n_before:,} tracts with missing features or "
      f"life expectancy, {len(tract):,} remaining")

X = tract[FEATURES]
y = tract[TARGET]

# Tract-level random folds, reused by every "5-fold CV" below (same mechanic as HW_03
# Part 3 and the Optuna companion notebook, so results are directly comparable).
rng = np.random.RandomState(RANDOM_STATE)
fold = pd.Series(rng.permutation(len(tract)) % 5)


def cv_mse(model_fn, fold=fold, **kwargs):
    """Average test MSE across the folds in `fold` (defaults to the shared 5-fold split)."""
    errs = []
    for f in sorted(fold.unique()):
        test_mask = (fold == f).values
        model = model_fn(**kwargs).fit(X[~test_mask], y[~test_mask])
        errs.append(mean_squared_error(y[test_mask], model.predict(X[test_mask])))
    return float(np.mean(errs))


def cv_tree_mse(**kwargs):
    return cv_mse(lambda **kw: DecisionTreeRegressor(random_state=RANDOM_STATE, **kw), **kwargs)


# The Art of Hyperparameter Tuning

For a good overview of everything in this notebook, see Bischl, Bernd, et al. "Hyperparameter optimization: Foundations, algorithms, best practices, and open challenges." *Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery* 13.2 (2023): e1484.

The goals of a supervised ML workflow fall into two categories:
1. Find the "best" model
2. Identify how good that model is

## Finding the best model

Very few machine learning approaches let you pick a method and be done. Most models have hyperparameters you need to select, and those choices matter a lot for performance.


In [ ]:

# Single 70/30 split, just for this illustration -- the CV-based tuning used
# everywhere else in this notebook comes later.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE)

depth_rows = []
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, depth in zip(axes, (2, 9, 20)):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, tree.predict(X_train))
    test_mse = mean_squared_error(y_test, tree.predict(X_test))
    depth_rows.append({
        "max_depth": depth, "leaves": tree.get_n_leaves(),
        "train_mse": train_mse, "test_mse": test_mse,
    })

    ax.scatter(y_test, tree.predict(X_test), s=4, alpha=0.3, color="#3b6fa0")
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, color="#888888", linewidth=1)
    ax.set_title(f"max_depth={depth} ({tree.get_n_leaves()} leaves)\n"
                 f"train MSE={train_mse:.2f}, test MSE={test_mse:.2f}")
    ax.set_xlabel("Actual life expectancy")
axes[0].set_ylabel("Predicted life expectancy")
plt.tight_layout()
plt.show()

pd.DataFrame(depth_rows)


### Hyperparameter tuning is a hard optimization problem

- Need to optimize along many different dimensions
- Hyperparameters typically interact with each other.

It's impossible to test every combination of hyperparameters, so in practice you only look at a subset of the full space.


#### Option 1: Tune one at a time
You can tune each parameter one at a time.
This is a bad idea usually:
- Hyperparameters interact
- Many local optima




In [ ]:

# Illustration: why tuning hyperparameters one at a time can fail when they're
# correlated. A synthetic two-hyperparameter objective with a diagonal ridge -- moving
# along the ridge barely changes the objective, moving across it changes it a lot.
def toy_objective(p1, p2):
    return (p1 - p2) ** 2 + 0.2 * (p1 + p2 - 6) ** 2  # minimized at p1 = p2 = 3


p1_grid = np.linspace(0, 6, 200)
p2_grid = np.linspace(0, 6, 200)
P1, P2 = np.meshgrid(p1_grid, p2_grid)
Z = toy_objective(P1, P2)

# Coordinate descent: alternate exact 1-D minimization of one hyperparameter at a
# time, holding the other fixed -- i.e. "tune each one individually".
path = [(0.0, 0.0)]
for _ in range(8):
    p1, p2 = path[-1]
    p1 = (2 / 3) * p2 + 1  # exact minimizer over p1 given p2
    path.append((p1, p2))
    p2 = (2 / 3) * p1 + 1  # exact minimizer over p2 given p1
    path.append((p1, p2))
path = np.array(path)

fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contourf(P1, P2, Z, levels=20, cmap="viridis_r")
plt.colorbar(cs, ax=ax, label="objective (lower is better)")
ax.plot(path[:, 0], path[:, 1], "o-", color="white", markersize=4, linewidth=1.5,
        label="tuning one hyperparameter at a time")
ax.scatter([3], [3], marker="*", s=250, color="red", label="true joint optimum", zorder=5)
ax.set_xlabel("hyperparameter 1")
ax.set_ylabel("hyperparameter 2")
ax.set_title("Correlated hyperparameters: a diagonal ridge")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()


#### Option 2: Grid Search

Create a grid of hyperparameters that you want to search through.

**Advantage**: If the grid has high enough resolution, you are very likely to get a near-optimal set of hyperparameters

**Disadvantages**: With more than 2-3 hyperparameters, it's usually too expensive.

In [ ]:

# A small, 2-dimensional grid: max_depth x min_samples_leaf, 5-fold CV on each combo.
depths = list(range(1, 8)) if FAST else list(range(1, 16))
leaf_sizes = [5, 50] if FAST else [1, 5, 20, 50, 100]

grid_mse = pd.DataFrame(
    {leaf: [cv_tree_mse(max_depth=d, min_samples_leaf=leaf) for d in depths] for leaf in leaf_sizes},
    index=depths,
)
grid_mse.index.name = "max_depth"

best_depth, best_leaf = grid_mse.stack().idxmin()
best_grid_mse = grid_mse.stack().min()
print(f"Best on this grid: max_depth={best_depth}, min_samples_leaf={best_leaf}, "
      f"CV test MSE={best_grid_mse:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(grid_mse.values, aspect="auto", cmap="viridis_r")
ax.set_xticks(range(len(leaf_sizes)))
ax.set_xticklabels(leaf_sizes)
ax.set_yticks(range(len(depths)))
ax.set_yticklabels(depths)
ax.set_xlabel("min_samples_leaf")
ax.set_ylabel("max_depth")
ax.set_title("5-fold CV test MSE: 2-parameter grid search")
plt.colorbar(im, ax=ax, label="CV test MSE")
ax.scatter([leaf_sizes.index(best_leaf)], [depths.index(best_depth)], marker="*", s=250, color="red")
plt.tight_layout()
plt.show()

# This grid is still cheap: 15 depths x 5 leaf sizes = 75 combinations, each a 5-fold CV
# fit. But grid search cost is *multiplicative* in the number of hyperparameters -- add
# a handful more and it stops being feasible:
#
# from itertools import product
# min_samples_splits = [2, 10, 50, 150, 400]           # 5 values
# ccp_alphas = np.logspace(-6, 0, 10)                   # 10 values, log-scale
# max_features_opts = [None, "sqrt", "log2"]            # 3 values
# for d, leaf, split, ccp, feats in product(depths, leaf_sizes, min_samples_splits,
#                                            ccp_alphas, max_features_opts):
#     ...  # 15 * 5 * 5 * 10 * 3 = 11,250 CV fits -- the curse of dimensionality in action


#### Option 3: Random Search

Strangely, random searches can outperform grid searches when there are many hyperparameters to tune.

#### Option 4: Fancier methods

- Bayesian optimization (e.g. Optuna's default TPE sampler) models which hyperparameters tend to score well and proposes the next trial where that model is most promising.
- Multi-fidelity methods like Hyperband or successive halving evaluate many configurations cheaply -- fewer rounds, a data subsample -- and only spend the full budget on the most promising ones.
- Evolutionary or genetic algorithms keep a population of configurations, retain the best performers, and mutate or recombine them into the next generation.

**Advantages**: Specifically designed to explore hyperparameter space more efficiently than the others.

**Disadvantages**: Hard to understand what's going on under the hood.

In [ ]:

# Same CART model, but tuning 5 hyperparameters jointly instead of 2 -- infeasible as an
# exhaustive grid (see the comment above), easy for Optuna.
def tree_objective(trial):
    params = dict(
        max_depth=trial.suggest_int("max_depth", 1, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 200, log=True),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 400, log=True),
        ccp_alpha=trial.suggest_float("ccp_alpha", 1e-6, 1.0, log=True),
        max_features=trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
    )
    return cv_tree_mse(**params)


N_TRIALS = 10 if FAST else 40
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(tree_objective, n_trials=N_TRIALS)

print(f"Grid search (2 hyperparameters)         : best CV test MSE={best_grid_mse:.3f}")
print(f"Optuna ({N_TRIALS} trials, 5 hyperparameters): best CV test MSE={study.best_value:.3f}")
print(f"Optuna best params: {study.best_params}")

# See Optuna_hyperparameter_tuning.ipynb for a closer look at what the search actually
# did: optimization history across trials, and which of the 5 hyperparameters mattered.


## Identifying how "good" that model is

There are three general approaches:
- Evaluate on single hold-out dataset
- Use the performance on the cross-validation holdout sets to assess performance
- Nested cross validation.

### Evaluation on a single hold-out dataset

**Advantages**:

- Simple
- Unbiased estimate of model performance
  
**Disadvantages:**

- Requires more data
- Model performance estimate has higher variance


### Re-using performance on cross-validation holdout sets

**Advantages**:

- Relatively simple
- Uses all the data to estimate model performance
- Relies on the same statistics used to optimize hyperparameters

**Disadvantages**:

- Biased (estimated error can be lower than the true error)

### Nested Cross-Validation

**Advantages**:

- Uses all the data to estimate model perfomance
- Estimate of error is unbiased

**Disadvantages**:

- More complicated to set up
- More computationally expensive



In [ ]:

# Toy comparison: three ways to estimate "how good" a tuned CART model really is, all
# tuning max_depth (min_samples_leaf=50 fixed) on the life-expectancy data.
toy_depths = list(range(1, 8)) if FAST else list(range(1, 16))


def cv_mse_on(Xs, ys, folds, **kwargs):
    errs = []
    for f in sorted(folds.unique()):
        test_mask = (folds == f).values
        model = DecisionTreeRegressor(random_state=RANDOM_STATE, **kwargs).fit(Xs[~test_mask], ys[~test_mask])
        errs.append(mean_squared_error(ys[test_mask], model.predict(Xs[test_mask])))
    return float(np.mean(errs))




# 1. CV on a training set + a single untouched holdout: split off a test set first, tune
#    max_depth with 5-fold CV using only the training rows, then evaluate the chosen
#    depth exactly once on the held-out rows it never saw during tuning.
X_tr, X_ho, y_tr, y_ho = train_test_split(X, y, test_size=1 / 3, random_state=RANDOM_STATE)
fold_tr = pd.Series(np.random.RandomState(RANDOM_STATE).permutation(len(X_tr)) % 5, index=X_tr.index)
holdout_mse_by_depth = [cv_mse_on(X_tr, y_tr, fold_tr, max_depth=d, min_samples_leaf=50) for d in toy_depths]
holdout_depth = toy_depths[int(np.argmin(holdout_mse_by_depth))]
holdout_tree = DecisionTreeRegressor(max_depth=holdout_depth, min_samples_leaf=50,
                                      random_state=RANDOM_STATE).fit(X_tr, y_tr)
holdout_estimate = mean_squared_error(y_ho, holdout_tree.predict(X_ho))


# 2. Traditional CV: tune max_depth with 5-fold CV, then report that same CV's own best
#    score as "the" error. Biased: the depth that won the search was the one that got
#    lucky on these particular folds, so its own CV score overstates how good it is.
traditional_mse = [cv_tree_mse(max_depth=d, min_samples_leaf=50) for d in toy_depths]
traditional_depth = toy_depths[int(np.argmin(traditional_mse))]
traditional_estimate = min(traditional_mse)

# 3. Nested CV: an outer 5-fold split provides 5 independent test sets. For each outer
#    fold, an *inner* 5-fold CV (on the outer-training rows only) picks max_depth, and
#    the resulting tree is scored once on the outer-test rows it never touched.
outer_scores, outer_best_depths = [], []
for outer_f in sorted(fold.unique()):
    outer_test_mask = (fold == outer_f).values
    X_out_tr, y_out_tr = X[~outer_test_mask], y[~outer_test_mask]
    X_out_te, y_out_te = X[outer_test_mask], y[outer_test_mask]

    inner_fold = pd.Series(
        np.random.RandomState(RANDOM_STATE + outer_f).permutation(len(X_out_tr)) % 5, index=X_out_tr.index
    )
    inner_mse = [cv_mse_on(X_out_tr, y_out_tr, inner_fold, max_depth=d, min_samples_leaf=50) for d in toy_depths]
    chosen_depth = toy_depths[int(np.argmin(inner_mse))]
    outer_best_depths.append(chosen_depth)

    outer_tree = DecisionTreeRegressor(max_depth=chosen_depth, min_samples_leaf=50,
                                        random_state=RANDOM_STATE).fit(X_out_tr, y_out_tr)
    outer_scores.append(mean_squared_error(y_out_te, outer_tree.predict(X_out_te)))

nested_estimate, nested_sd = np.mean(outer_scores), np.std(outer_scores)

print(f"1. CV + single holdout : depth={holdout_depth:>2}, held-out test MSE={holdout_estimate:.3f}")
print(f"2. Traditional CV      : depth={traditional_depth:>2}, reported test MSE={traditional_estimate:.3f}")
print(f"3. Nested CV           : depth chosen per outer fold={outer_best_depths}")
print(f"                         mean outer test MSE={nested_estimate:.3f} +/- {nested_sd:.3f}")


Schematic: nested CV. Three outer resampling iterations (I, II, III) rotate which third of the data is $\mathcal{D}^{(\text{outer})}_{\text{test}}$, used only for evaluation. The remaining two thirds ($\mathcal{D}^{(\text{outer})}_{\text{train}}$) feed an *inner* CV loop that proposes and evaluates hyperparameter configurations (HPC), and the tuned model from that inner loop is what actually predicts on the outer test fold.

Source: Bischl, Bernd, et al. "Hyperparameter optimization: Foundations, algorithms, best practices, and open challenges." *Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery* 13.2 (2023): e1484.

In [ ]:
from IPython.display import Image

Image(filename="demos/nested_cv_schematic.png", embed=True)


Estimated model error is itself a "random" quantity -- it depends on the specific training set, the specific test set, and the random splits used to compute it.

In [ ]:

# Same fixed-hyperparameter tree, scored under many different random splits/folds, to
# see how much the "error estimate" itself varies just from the luck of the draw.
N_SEEDS = 8 if FAST else 30
seed_rows = []
for seed in range(N_SEEDS):
    Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X, y, test_size=1 / 3, random_state=seed)
    tree = DecisionTreeRegressor(max_depth=7, min_samples_leaf=50, random_state=RANDOM_STATE).fit(Xs_tr, ys_tr)
    holdout_mse = mean_squared_error(ys_te, tree.predict(Xs_te))

    seed_fold = pd.Series(np.random.RandomState(seed).permutation(len(X)) % 5)
    cv_mse_seed = cv_tree_mse(fold=seed_fold, max_depth=7, min_samples_leaf=50)

    seed_rows.append({"seed": seed, "holdout_mse": holdout_mse, "cv_mse": cv_mse_seed})

seed_df = pd.DataFrame(seed_rows)
print(f"Single 1/3 holdout : mean={seed_df['holdout_mse'].mean():.3f}, sd={seed_df['holdout_mse'].std():.3f}")
print(f"5-fold CV          : mean={seed_df['cv_mse'].mean():.3f}, sd={seed_df['cv_mse'].std():.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([seed_df["holdout_mse"], seed_df["cv_mse"]], tick_labels=["single 1/3 holdout", "5-fold CV"])
ax.set_ylabel("Test MSE (life expectancy, years$^2$)")
ax.set_title(f"Same max_depth=7 tree, {N_SEEDS} different random seeds")
plt.tight_layout()
plt.show()


The "best" model might not actually be the best model. Often selecting a simpler model within the uncertainty of the best model is the way to go.

## Demo: Putting this all together with XGBoost

In [ ]:

# 50% train / 25% validation / 25% test. Optuna tunes XGBoost by fitting once on the
# training split and scoring once on the validation split per trial -- no 5-fold CV --
# which is much faster. The test split stays untouched until the very end.
X_tv, X_test_f, y_tv, y_test_f = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)
X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(X_tv, y_tv, test_size=1 / 3, random_state=RANDOM_STATE)


def xgb_val_mse(**params):
    model = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    model.fit(X_train_f, y_train_f)
    return mean_squared_error(y_val_f, model.predict(X_val_f))


def xgb_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-6, 10.0, log=True),
    )
    return xgb_val_mse(**params)


N_TRIALS = 40 if FAST else 100
study_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
t0 = time.time()
study_xgb.optimize(xgb_objective, n_trials=N_TRIALS)
print(f"tuned {N_TRIALS} trials in {time.time() - t0:.0f}s")

# Refit on train + validation combined now that hyperparameters are chosen, then
# evaluate -- for the first time -- on the untouched test split.
final_model = xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, **study_xgb.best_params)
final_model.fit(X_tv, y_tv)
true_test_mse = mean_squared_error(y_test_f, final_model.predict(X_test_f))

print(f"Validation estimate during tuning (best of {N_TRIALS} trials) : {study_xgb.best_value:.3f}")
print(f"True test MSE of that same model                              : {true_test_mse:.3f}")
print(f"Gap (optimism of the winning validation score)                : {true_test_mse - study_xgb.best_value:+.3f}")
